# sanskritGPT — Training Notebook

Checkpoints are saved to `/kaggle/output/out-sanskrit/` every 250 steps and are downloadable from the Output tab.

In [ ]:
# Cell 1 — verify GPU
!nvidia-smi

In [ ]:
# Cell 2 — install dependencies
!pip install transformers wandb -q

In [ ]:
# Cell 3 — clone repo
import os
if not os.path.exists('/kaggle/working/sanskritGPT'):
    !git clone https://github.com/vedchamp07/sanskritGPT.git
os.chdir('/kaggle/working/sanskritGPT')
print('Working directory:', os.getcwd())

# make sure output dir exists
os.makedirs('/kaggle/output/out-sanskrit', exist_ok=True)
print('Output dir ready: /kaggle/output/out-sanskrit')

In [ ]:
# Cell 4 — symlink dataset
import os, glob
os.chdir('/kaggle/working/sanskritGPT')

bins = glob.glob('/kaggle/input/**/train.bin', recursive=True)
assert len(bins) > 0, 'train.bin not found — attach sanskrit-gpt-data dataset'
data_path = bins[0].rsplit('/', 1)[0]
print('Dataset found at:', data_path)

os.makedirs('data/sanskrit', exist_ok=True)
for f in ['train.bin', 'val.bin', 'meta.pkl']:
    src = f'{data_path}/{f}'
    dst = f'data/sanskrit/{f}'
    if os.path.lexists(dst):
        os.remove(dst)
    os.symlink(src, dst)
    print(f'  linked {f}')

!ls -lh data/sanskrit/

In [ ]:
# Cell 5 — verify data
import os, numpy as np
os.chdir('/kaggle/working/sanskritGPT')

train = np.memmap('data/sanskrit/train.bin', dtype=np.uint32, mode='r')
val   = np.memmap('data/sanskrit/val.bin',   dtype=np.uint32, mode='r')
print(f'Train tokens : {len(train):,}')
print(f'Val tokens   : {len(val):,}')
print(f'Max token id : {train.max()}')
print(f'Out of range : {(train >= 68096).any()}')
assert not (train >= 68096).any(), 'Token ids out of vocab range!'
print('Data looks good.')

In [ ]:
# Cell 6 — setup wandb
import os
os.chdir('/kaggle/working/sanskritGPT')
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['WANDB_API_KEY'] = UserSecretsClient().get_secret('WANDB_API_KEY')
    !wandb login $WANDB_API_KEY --relogin
    print('wandb ready')
except Exception as e:
    print(f'wandb skipped: {e}')

In [ ]:
# Cell 7 — train
# Checkpoints save to /kaggle/output/sanskrit-gpt-out/ every 250 steps
import os
os.chdir('/kaggle/working/sanskritGPT')
!python train.py config/train_sanskrit.py --wandb_log=True --out_dir=/kaggle/output/sanskrit-gpt-out

In [ ]:
# Cell 8 — verify checkpoint saved to output
import os
print('Output directory contents:')
!ls -lh /kaggle/output/sanskrit-gpt-out/
print()
print('losses.csv preview:')
!head -20 /kaggle/output/sanskrit-gpt-out/losses.csv 2>/dev/null || echo 'no losses.csv found'

In [ ]:
# Cell 9 — copy final checkpoint explicitly (safety net)
import os, shutil
ckpt_src = '/kaggle/output/out-sanskrit/ckpt.pt'
ckpt_dst = '/kaggle/output/ckpt.pt'
if os.path.exists(ckpt_src):
    shutil.copy(ckpt_src, ckpt_dst)
    size = os.path.getsize(ckpt_dst) / 1e6
    print(f'Checkpoint copied to /kaggle/output/ckpt.pt ({size:.1f} MB)')
else:
    print('ERROR: ckpt.pt not found at', ckpt_src)
    print('All files in output:')
    for root, dirs, files in os.walk('/kaggle/output'):
        for f in files:
            path = os.path.join(root, f)
            print(f'  {path} ({os.path.getsize(path)/1e6:.1f} MB)')

In [ ]:
# Cell 10 — sample from trained model
import os
os.chdir('/kaggle/working/sanskritGPT')
!python sample.py --out_dir=/kaggle/output/out-sanskrit --start='नमस्ते' --num_samples=3 --max_new_tokens=100